In [1]:
# Loading the credentials from the env file
from gen_ai_hub.proxy.gen_ai_hub_proxy import GenAIHubProxyClient
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# Fetching environment variables
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")
AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")

# Initializing the GenAIHubProxyClient
client = GenAIHubProxyClient(
    base_url=AICORE_BASE_URL,
    auth_url=AICORE_AUTH_URL,
    client_id=AICORE_CLIENT_ID,
    client_secret=AICORE_CLIENT_SECRET,
    resource_group=AICORE_RESOURCE_GROUP
)


# Dependencies and Helper Functions

In [2]:
!pip install rich PyYAML "sap-ai-sdk-gen[all]"

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python3.13 -m pip install --upgrade pip


In [3]:
from gen_ai_hub.proxy import get_proxy_client
import pathlib
import yaml

from ai_api_client_sdk.models.input_artifact_binding import InputArtifactBinding
from ai_api_client_sdk.models.parameter_binding import ParameterBinding
from ai_api_client_sdk.models.artifact import Artifact
from ai_api_client_sdk.models.label import Label

SUPPORTED_METRICS = [
    "LLMaaJ:Sem_Sim_1",    # LLM-as-a-judge semantic similarity, binary
    "LLMaaJ:Sem_Sim_10",   # LLM-as-a-judge semantic similarity, scale 1-10
    "JSON_Match",           # JSON field-level precision/recall/f1
    "FC:ExactMatch",        # Exact match for function calling use cases
    "BLEU",                 # N-gram precision for translation quality
    "ROUGE",                # N-gram overlap for summarization/translation
    "METEOR",               # Precision/recall with synonym matching
    "EXACT_MATCH",          # Boolean exact match
]
# For custom LLM-as-a-judge metrics, use custom_metric_id in create_config instead.
# Note: Only custom metrics with the evaluation method LLM-as-a-judge and numerical
# or Boolean output types can be included in prompt optimizations.
# See: https://help.sap.com/docs/sap-ai-core/generative-ai/custom-metrics

In [4]:
client = get_proxy_client()

In [5]:
from logging import PlaceHolder
from pydantic import BaseModel
from typing import List
import re
import requests
import json

class PromptTemplate(BaseModel):
    role: str
    content: str


class PromptTemplateSpec(BaseModel):
    template: List[PromptTemplate]


    @property
    def placeholders(self):
        placeholders = set()
        pattern = re.compile(r'\{\{\s*\?\s*(\w+)\s*\}\}')
        for message in self.template:
            placeholders.update(pattern.findall(message.content))
        return placeholders

    @classmethod
    def from_optimizer_result(cls, input_):
        placeholders = input_["user_message_template_fields"]
        def replace(msg):
            for key in placeholders:
                msg = msg.replace("{"+key+"}", "{{?"+ key + "}}")
            return msg

        return cls(
            template=[
                {
                    "role": "system",
                    "content": replace(input_["system_prompt"]),
                },{
                    "role": "user",
                    "content": replace(input_["user_message_template"]),
                }
            ]
        )

    def escape_curly_brackets(self) -> "PromptTemplateSpec":
        # 1. Hide each {{?key}} placeholder with a unique token
        placeholder_pattern = re.compile(r'\{\{\s*\?\s*(\w+)\s*\}\}')
        mapping = {}
        counter = 1

        def _hide(match):
            nonlocal counter
            token = f"__PLACEHOLDER_{counter}__"
            mapping[token] = match.group(0)
            counter += 1
            return token

        new_templates = []
        for msg in self.template:
            # a) hide custom placeholders
            hidden = placeholder_pattern.sub(_hide, msg.content)
            # b) escape all remaining braces
            escaped = hidden.replace('{', '{{').replace('}', '}}')
            # c) restore the original placeholders
            print(mapping)
            for token, original in mapping.items():
                escaped = escaped.replace(token, original)

            new_templates.append(PromptTemplate(role=msg.role, content=escaped))

        # return a fresh copy
        return PromptTemplateSpec(template=new_templates)



def fetch_prompt_template(prompt_template: str) -> PromptTemplateSpec:
    headers = {
        **client.request_header,
        "Content-Type": "application/json",
    }
    url = f"{client.ai_core_client.base_url}/lm/promptTemplates"
    scenario, sep, name = prompt_template.partition("/")
    if sep:
        name, sep, version = name.partition(":")
    if sep:
        body = {"name": name,
                "version": version,
                "scenario": scenario,
                "includeSpec": True
            }
        response =  requests.get(url, headers=headers, params=body)
        response.raise_for_status()
        response = response.json()
        if response["count"] > 0:
            response = response["resources"][0]
        else:
            raise ValueError(f"Prompt template {name} not found.")
    else:
        url += f"/{prompt_template}"
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        response = response.json()
    return PromptTemplateSpec.model_validate(response["spec"])

def load_prompt_template(prompt: str | pathlib.Path | list | dict | PromptTemplateSpec) -> PromptTemplateSpec:
    if isinstance(prompt, PromptTemplateSpec):
        return prompt
    if isinstance(prompt, (str, pathlib.Path)) and pathlib.Path(prompt).exists():
        with open(prompt, "r") as f:
            prompt = yaml.safe_load(f)
    elif isinstance(prompt, str):
        return fetch_prompt_template(prompt)
    if isinstance(prompt, dict):
        # expect dict with keys "system" [optional] and "user"
        messages = []
        if "system" in prompt:
            messages.append({"role": "system", "content": prompt["system"]})
        messages.append({"role": "user", "content": prompt["user"]})
        return PromptTemplateSpec(template=messages)
    elif isinstance(prompt, list):
        # expect list of dicts with keys "role" and "content"
        return PromptTemplateSpec(template=prompt)
    else:
        raise ValueError("Prompt must be a string, Path, list or dict")


def push_prompt_template(prompt_template: PromptTemplateSpec,
                         prompt_template_name_registry: str,
                         prompt_template_version: str,
                         scenario: str,
                         update=False):
    headers = {
        **client.request_header,
        "Content-Type": "application/json",
    }
    url = f"{client.ai_core_client.base_url}/lm/promptTemplates"
    body = {"name": prompt_template_name_registry,
            "version": prompt_template_version,
            "scenario": scenario}
    res = requests.get(url, headers=headers, params=body).json()
    if res["count"] > 0 and not update:
        print(f"Prompt template {prompt_template_name_registry} already exists. Use update=True to update.")
        return res["resources"][0]
    # Prepare body

    body["spec"] = prompt_template.model_dump()
    # Prepare headers
    response = requests.post(url, headers=headers, json=body)
    # Handle response
    if response.status_code == 201:
        response = response.json()
    elif response.status_code in (400, 409, 413):
        # Return error details
        raise requests.HTTPError(f"Upload failed ({response.status_code}): {response.text}")
    else:
        response.raise_for_status()
    return response.json()


import re

def convert_py_notation(template):
    pattern = re.compile(r'\{\{\s*\?\s*(\w+)\s*\}\}')
    return pattern.sub(lambda match: "{" + match.group(1) + "}", template)


def validate_prompt(prompt: PromptTemplateSpec):
    values = {k: "???" for k in prompt.placeholders}

    for message in prompt.template:
        if message.role == "user":
            try:
                convert_py_notation(message.content).format(**values)
            except KeyError as err:
                msg = ["Unexpected key error when running test formatting."]
                msg += ["This is most likeyly due to unescaped curly brackets."]
                msg += ["You can try fixing this by running `prompt = prompt.escape_curly_brackets()` and use the new prompt template."]
                raise ValueError("\n".join(msg)) from err
    return True




from rich.console import Console
from rich.highlighter import RegexHighlighter
from rich.theme import Theme
from rich.panel import Panel
from rich import print

class TemplateHighlighter(RegexHighlighter):
    """Apply style to anything that looks like an email."""

    base_style = "template."
    highlights = [r"(?P<placeholder>\{\{\s*\?[^\{\}\s]+\s*\}\})"]

highlighter = TemplateHighlighter()
theme = Theme({"template.placeholder": "bold magenta", "example.email": "bold magenta"})
console = Console(highlighter=highlighter, theme=theme)


def print_prompt_template(prompt_template: PromptTemplateSpec | str | pathlib.Path, addition: str | None = None):

    prompt_template = load_prompt_template(prompt_template)
    addition = f' - {addition}' if addition else ''

    for message in prompt_template.template:
        if message.role == "system":
            console.print(Panel(highlighter(message.content), title="System Message" + addition, border_style="red"))
        elif message.role == "user":
            console.print(Panel(highlighter(message.content), title="User Message" + addition, border_style="green"))
        else:
            console.print(Panel(highlighter(message.content), title="Assistant Message" + addition))



In [6]:
from typing import List
import requests
import mimetypes
from urllib.parse import quote
import pathlib
import json


def validate_dataset(dataset: str | pathlib.Path | list, expected_keys: None | List[str] = None) -> bool:
    if isinstance(dataset, (str, pathlib.Path)):
        with open(dataset, "r") as f:
            try:
                dataset = json.load(f)
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON in file: {e}")
    if not isinstance(dataset, list):
        raise ValueError("Dataset must be a list of dictionaries.")

    def validate_item(item: dict, excepted_keys: None | List[str]) -> bool:
        excepted_keys = set(excepted_keys) if excepted_keys else None
        if set(item.keys()) != {"fields", "answer"}:
            raise ValueError("Each item must contain 'fields' and 'answer' keys.")
        if not isinstance(item["fields"], dict):
            raise ValueError("'fields' must be a dictionary.")
        fields = set(item["fields"].keys())
        if excepted_keys is not None:
            if fields != excepted_keys:
                if fields.difference(excepted_keys):
                    raise ValueError(f"Unexpected keys in 'fields'. Expected: {excepted_keys}, Found: {fields}")
                if excepted_keys.difference(fields):
                    raise ValueError(f"Missing keys in 'fields'. Expected: {excepted_keys}, Found: {fields}")
        if not all([isinstance(k, str) for k in item["fields"].values()]):
            raise ValueError("All values in 'fields' must be strings.")
        return fields

    excepted_keys = expected_keys
    for i, item in enumerate(dataset):
        if not isinstance(item, dict):
            raise ValueError("Each item in the dataset must be a dictionary.")
        try:
            excepted_keys = validate_item(item, excepted_keys)
        except ValueError as e:
            raise ValueError(f"Error in entry {i}") from e
    return True


def upload_dataset(secret: str,
                   local_path: str | pathlib.Path,
                   remote_path: str,
                   scenario: str,
                   description: str | None = None,
                   overwrite: bool = False,
                   expected_keys: None | List[str] = None,

                   allow_bucket_root: bool = False) -> str:
    # Validate dataset
    validate_dataset(local_path, expected_keys)
    # check if secret exists
    secrets = [r.name for r in client.ai_core_client.object_store_secrets.query().resources]
    if secret not in secrets:
        raise ValueError(f"Secret '{secret}' not found in object store secrets. Known secrets: {secrets}")

    # Check if local path exists
    remote_path = remote_path.lstrip("/")
    if "/" not in remote_path and not allow_bucket_root:
        raise ValueError(
            "Remote path must use subdirectories. Otherwise the whole bucket will be used as an input artifact. Set allow_bucket_root=True to allow this."
        )

    # URL-encode the path parameter
    path = f"{secret}/" + remote_path.lstrip("/")
    encoded_path = quote(path, safe="")
    url = f"{client.ai_core_client.base_url}/lm/dataset/files/{encoded_path}"
    params = {"overwrite": str(overwrite).lower()}

    # Prepare headers
    headers = {
        **client.request_header,
        "Content-Type": "application/octet-stream",
    }
    # Guess MIME type
    guessed_type, _ = mimetypes.guess_type(local_path)
    if guessed_type:
        headers["Content-Type"] = guessed_type

    with open(local_path, "rb") as f:
        response = requests.put(url, params=params, headers=headers, data=f)

    # Handle response
    if response.status_code == 201:
        response = response.json()
    elif response.status_code in (400, 409, 413):
        # Return error details
        raise requests.HTTPError(f"Upload failed ({response.status_code}): {response.text}")
    else:
        response.raise_for_status()
    artifact_url = "/".join(response["url"].split("/")[:-1])
    for artifact in client.ai_core_client.artifact.query().resources:
        if response["url"].startswith(artifact.url + "/"):
            return artifact, response["url"].removeprefix(artifact.url).lstrip("/")

    # Create new artifact
    path = response["url"].split("/")[-1]
    new_artifact = client.ai_core_client.artifact.create(
        name=f"{scenario}-prompt-optimization",
        kind=Artifact.Kind.DATASET,
        url=artifact_url,
        scenario_id=scenario,
        description="Datasets for prompt optimization" if description is None else description,
        resource_group=headers[client.ai_core_client.rest_client.resource_group_header]
    )
    return new_artifact, path



## Create Config

In [7]:
def create_config(metric: str = None,
                  custom_metric_id: str = None,
                  reference_model: str = None,
                  targets: dict = None,
                  dataset_path: str = None,
                  scenario: str = None,
                  prompt: dict = None,
                  artifact_id: str = None,
                  model_params: str = None,
                  variable_mapping: str = None,
                  prototype_mode: str = "false",
                  field_evaluation_metrics: dict = None) -> str:
    assert metric or custom_metric_id, "Provide either metric or custom_metric_id"
    assert not (metric and custom_metric_id), "Provide only one of metric or custom_metric_id"
    if metric:
        assert metric in SUPPORTED_METRICS, f"Unsupported metric: {metric}. Supported: {SUPPORTED_METRICS}"
    assert artifact_id, "artifact_id is required"

    input_parameters = [
        ParameterBinding(key="dataset", value=dataset_path),
        ParameterBinding(key="optimizationMetric", value=metric or "none"),
        ParameterBinding(key="customMetricId", value=custom_metric_id or "none"),
        ParameterBinding(key="basePrompt", value=f'{scenario}/{prompt["name"]}:{prompt["version"]}'),
        ParameterBinding(key="baseModel", value=reference_model),
        ParameterBinding(key="targetModels", value=','.join(targets.keys())),
        ParameterBinding(key="targetPromptMapping", value=",".join([f"{k}={v}" for k, v in targets.items()])),
        ParameterBinding(key="modelParams", value=model_params or "none"),
        ParameterBinding(key="variableMapping", value=variable_mapping or "none"),
        ParameterBinding(key="prototypeMode", value=prototype_mode),
        ParameterBinding(key="fieldEvaluationMetrics", value=json.dumps(field_evaluation_metrics) if field_evaluation_metrics else "none"),
    ]
    existing_configs = client.ai_core_client.configuration.query(scenario_id='genai-optimizations', executable_ids=['genai-optimizations'])
    params = {par.key: par.value for par in input_parameters}
    for conf in existing_configs.resources:
        if {par.key: par.value for par in conf.parameter_bindings} == params:
            return conf.id

    input_artifacts = [InputArtifactBinding(key="prompt-data", artifact_id=artifact_id)]

    response = client.ai_core_client.configuration.create(
        name="prompt-optimization-config",
        scenario_id="genai-optimizations",
        executable_id="genai-optimizations",
        resource_group=resource_group,
        parameter_bindings=input_parameters,
        input_artifact_bindings=input_artifacts
    )
    return response.id


In [8]:
import json
from ai_core_sdk.tracking import Tracking
from ai_api_client_sdk.models.metric_resource import MetricResource
from rich.table import Table
from rich import print


def _fetch_aggregations(execution_id):
    tracking_client = Tracking(
        base_url=client.ai_core_client.base_url,
        token_creator=client.ai_core_client.rest_client.get_token,
        resource_group=resource_group,
    )
    result = tracking_client.query(
        execution_ids=[execution_id],
        resource_group=resource_group,
    )
    path = f"/lm/metrics?tagFilters=evaluation.ai.sap.com/child-of={execution_id}"
    child_response = client.ai_core_client.rest_client.get(
        path=path,
        resource_group=resource_group,
    )
    child_resources = [
        MetricResource.from_dict(r) for r in child_response.get("resources", [])
    ]
    result.resources = (result.resources or []) + child_resources
    return result


def _parse_custom_val(custom_val):
    try:
        return json.loads(custom_val.value)
    except (json.JSONDecodeError, TypeError):
        return custom_val.value


def _collect_result_data(aggregations):
    models = {}
    for resource in (aggregations.resources or []):
        tags = {t.name: t.value for t in (resource.tags or [])}
        model = tags.get("evaluation.ai.sap.com/model") or resource.execution_id
        purpose = tags.get("evaluation.ai.sap.com/purpose", "unknown")
        entry = models.setdefault(model, {})

        if purpose == "origin":
            for metric in (resource.metrics or []):
                entry.setdefault("baseline", metric.value)
            for cv in (resource.custom_info or []):
                val = _parse_custom_val(cv)
                if cv.name == "origin_model_evals":
                    entry["baseline_eval"] = val
                elif cv.name == "origin_llm_request_metrics":
                    entry["tokens"] = val
                else:
                    entry.setdefault("custom", {})[cv.name] = val

        elif purpose == "target":
            prompt_id = tags.get("evaluation.ai.sap.com/promptTemplateId")
            if prompt_id:
                entry["prompt_template_id"] = prompt_id
            for metric in (resource.metrics or []):
                label = next(
                    (lbl.value for lbl in (metric.labels or []) if lbl.name == "optimizer_metric_type"),
                    None,
                )
                if label:
                    if "pre" in label.lower():
                        entry["pre"] = metric.value
                    elif "post" in label.lower():
                        entry["post"] = metric.value
            for cv in (resource.custom_info or []):
                val = _parse_custom_val(cv)
                if cv.name == "pre_optimization_evaluation":
                    entry["pre_eval"] = val
                elif cv.name == "post_optimization_evaluation":
                    entry["post_eval"] = val
                elif cv.name == "llm_request_metrics":
                    entry["tokens"] = val
                else:
                    entry.setdefault("custom", {})[cv.name] = val
    return models


def _fetch_optimized_prompt(prompt_template_id):
    url = f"{client.ai_core_client.base_url}/lm/promptTemplates/{prompt_template_id}"
    response = requests.get(url, headers={**client.request_header})
    response.raise_for_status()
    return PromptTemplateSpec.model_validate(response.json()["spec"])


def fetch_and_print_results(execution_id):
    status = client.ai_core_client.execution.get(execution_id=execution_id).status
    if status.name not in {"COMPLETED", "DEAD"}:
        print(f"[yellow]Execution is still {status.name}. Wait for it to complete.[/yellow]")
        return

    aggregations = _fetch_aggregations(execution_id)
    data = _collect_result_data(aggregations)

    if not data:
        print("[red]No results found for this execution.[/red]")
        return

    def fmt(v):
        if v is None: return "–"
        return f"{v:.3f}" if isinstance(v, float) else f"{v:,}" if isinstance(v, int) else str(v)

    # Score summary
    score_table = Table(title="Score Summary")
    score_table.add_column("Model", style="cyan")
    score_table.add_column("Baseline", justify="center", style="white")
    score_table.add_column("Pre", justify="center", style="magenta")
    score_table.add_column("Post", justify="center", style="green")
    score_table.add_column("Improvement", justify="center", style="bold")

    for model, d in data.items():
        pre, post, baseline = d.get("pre"), d.get("post"), d.get("baseline")
        if pre is None and post is None and baseline is None:
            continue
        if pre is not None and post is not None and pre != 0:
            delta = (post - pre) / pre * 100
            improvement = f"{'▲' if delta >= 0 else '▼'} {delta:+.1f}%"
        elif pre is not None and post is not None:
            diff = post - pre
            improvement = f"{'▲' if diff >= 0 else '▼'} {diff:+.3f}"
        else:
            improvement = "–"
        score_table.add_row(
            model,
            f"{baseline:.3f}" if baseline is not None else "–",
            f"{pre:.3f}" if pre is not None else "–",
            f"{post:.3f}" if post is not None else "–",
            improvement,
        )
    console.print(score_table)

    # Per-model detail
    for model, d in data.items():
        baseline_eval = d.get("baseline_eval") if isinstance(d.get("baseline_eval"), dict) else None
        pre_eval = d.get("pre_eval") if isinstance(d.get("pre_eval"), dict) else None
        post_eval = d.get("post_eval") if isinstance(d.get("post_eval"), dict) else None

        if baseline_eval or pre_eval or post_eval:
            eval_table = Table(title=f"Evaluation Details — {model}")
            eval_table.add_column("Metric", style="cyan")
            if baseline_eval:
                eval_table.add_column("Baseline", justify="right", style="white")
            if pre_eval:
                eval_table.add_column("Pre", justify="right", style="magenta")
            if post_eval:
                eval_table.add_column("Post", justify="right", style="green")
            keys = list(dict.fromkeys(
                list(baseline_eval.keys() if baseline_eval else []) +
                list(pre_eval.keys() if pre_eval else []) +
                list(post_eval.keys() if post_eval else [])
            ))
            for k in keys:
                row = [k]
                if baseline_eval:
                    row.append(fmt(baseline_eval.get(k)))
                if pre_eval:
                    row.append(fmt(pre_eval.get(k)))
                if post_eval:
                    row.append(fmt(post_eval.get(k)))
                eval_table.add_row(*row)
            console.print(eval_table)

        # Token consumption
        tokens = d.get("tokens")
        if isinstance(tokens, dict):
            tok_table = Table(title=f"Token Consumption — {model}")
            tok_table.add_column("Metric", style="cyan")
            tok_table.add_column("Value", justify="right", style="yellow")
            for k, v in tokens.items():
                tok_table.add_row(k, f"{v:,}" if isinstance(v, int) else str(v))
            console.print(tok_table)

        # Optimized prompt from registry
        prompt_template_id = d.get("prompt_template_id")
        if prompt_template_id:
            try:
                optimized_prompt = _fetch_optimized_prompt(prompt_template_id)
                print_prompt_template(optimized_prompt, addition=f"Optimized — {model}")
            except Exception as e:
                print(f"[yellow]Could not fetch optimized prompt for {model}: {e}[/yellow]")


### Download Demo Data

In [9]:
import pathlib

files = [
    ("default/example/base-prompt.yaml", "./facility_prompt.yaml"),
    ("default/example/facility-train.json", "./facility-train.json")
]

for remote, local in files:
    local_path = pathlib.Path(local)
    if not local_path.exists():
        url = f"{client.ai_core_client.base_url}/lm/dataset/files/{remote}"
        headers = {
            **client.request_header,
        }
        response = requests.get(url, headers=headers)
        with local_path.open("w") as stream:
            stream.write(response.text)

In [10]:
resource_group = client.request_header[client.ai_core_client.rest_client.resource_group_header]

# Start Prompt Optimizer Run

### Loading a Local Prompt Template

**The prompt template is structured in a `system` and a `user` message. Placeholders in the prompt template have to be wrapped in `{{?key}}`.**

Your prompt can be provided in any of the following forms and will be normalized to a `PromptTemplateSpec` under the hood:

#### From Local Disk
**A file path** (`str` or `Path`) pointing to a YAML or JSON file defining either:
  - a **mapping** with keys `"user"` (required) and `"system"` (optional)

```yaml
system: |-
    You are a helpful assistant
user: |-
    Write a poem on {{?topic}}
```
or a **list** of message objects with `"role"` and `"content"` keys.

#### Alternative: Prompt Registry
- **A lookup string** of the form `"<scenario>/<name>:<version>"` will be fetched from the AI Core prompt-template API.

#### Response Format (optional)
You can define a schema for the response structure to follow. To do this, include a `response_format` field in your prompt template and populating it with details of your structure.

To enable per-field evaluation metrics (`field_evaluation_metrics`), your prompt template must include a `response_format` in its spec. Add it to your YAML under the `spec` key alongside `template`:

```yaml
spec:
  template:
    - role: system
      content: |-
        You are a helpful assistant
    - role: user
      content: |-
        {{?input}}
  response_format:
    type: json_schema
    json_schema:
      name: facility_output
      strict: true
      schema:
        type: object
        properties:
          urgency:
            type: string
            enum: [high, medium, low]
          sentiment:
            type: string
            enum: [positive, negative, neutral]
          categories:
            type: object
            properties:
              emergency_repair_services:
                type: boolean
              routine_maintenance_requests:
                type: boolean
              quality_and_safety_concerns:
                type: boolean
              # more categories can be added here
            required:
              - emergency_repair_services
              - routine_maintenance_requests
              - quality_and_safety_concerns
              # add more required entries as needed
            additionalProperties: false
        required: [urgency, sentiment, categories]
        additionalProperties: false
```

In [11]:
base_prompt_template = "./facility_prompt.yaml" # local path to the prompt template or Prompt Repository identifier


prompt = load_prompt_template(base_prompt_template) # .escape_curly_brackets() if validation fails.
print_prompt_template(prompt)
print(f"Prompt template loaded successfully. Placeholders found are: {prompt.placeholders}")
assert validate_prompt(prompt)


╭──────────────────────────────────────────────── System Message ─────────────────────────────────────────────────╮
│ You are a helpful assistant                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── User Message ──────────────────────────────────────────────────╮
│ Giving the following message:                                                                                   │
│ ---                                                                                                             │
│ {{?input}}                                                                                                      │
│ ---                                                                                                             │
│ Extract and return a json with the follwoing keys and values:                                                   │
│ - "urgency" as one of `high`, `medium`, `low`                                                                   │
│ - "sentiment" as one of `negative`, `neutral`, `positive`                                                       │
│ - "categories" Create a dictionary with categories as keys and boolean values (True/False), where the value     │
│ indicates whether the category is one of the best matching support category tags from:                          │
│ `emergency_repair_services`, `routine_maintenance_requests`, `quality_and_safety_concerns`,                     │
│ `specialized_cleaning_services`, `general_inquiries`, `sustainability_and_environmental_practices`,             │
│ `training_and_support_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`,            │
│ `facility_management_issues`                                                                                    │
│ Your complete message should be a valid json string that can be read directly and only contain the keys         │
│ mentioned in the list above. Never enclose it in ```json...```, no newlines, no unnessacary whitespaces.        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Prompt template loaded successfully. Placeholders found are: {'input'}

Check if all expected placeholders were found.

### Validating Local Dataset

Your dataset must be a JSON‐serializable list where each element is a dictionary with exactly two keys: **`fields`** and **`answer`**. The **`fields`** value should itself be a dictionary whose keys (e.g. `"question"`, `"hint"`, `"term"`, etc.) are **identical** across every entry and whose values are all strings. The **`answer`** value must also be a string.


You can validate your dataset using the `validate_dataset` method.

If validation is not passed succesfully this are might be the reasons:


| Condition                              | Exception Raised (inner)                                           | Outer Message                    |
| -------------------------------------- | ------------------------------------------------------------------ | -------------------------------- |
| Non-list top-level                     | N/A                                                                | `Dataset must be a list…`        |
| Item not a dict                        | N/A                                                                | `Each item…must be a dictionary` |
| Wrong item keys                        | `ValueError("Each item must contain 'fields' and 'answer' keys.")` | `Error in entry i`               |
| `"fields"` not a dict                  | `ValueError("'fields' must be a dictionary.")`                     | `Error in entry i`               |
| Field name mismatch (extra or missing) | `ValueError("Unexpected keys…")` or `ValueError("Missing keys…")`  | `Error in entry i`               |
| Non-string field value                 | `ValueError("All values in 'fields' must be strings.")`            | `Error in entry i`               |
| Invalid JSON file                      | `ValueError("Invalid JSON in file:…")`                             | N/A                              |


In [12]:
dataset_local_path="./facility-json.json" # local path to the dataset

assert validate_dataset(dataset_local_path), "Dataset not valid"

### Remaining Config parameter

In [13]:
scenario = "genai-optimizations"

base_prompt_template_registry = "evaluate-base:0.0.1"  # name:version for the template in the registry

dataset_secret="default" # secret name in the object store you want to use to store the dataset
dataset_remote_path="datasets/facility-train.json" # remote path in the object store to store the dataset

reference_model = "gpt-4o:2024-08-06"
# Dictionary of models to optimize with their corresponding prompt template names under which the optimized prompt should be stored in the registry
targets = {
    "gemini-2.5-pro:001": "evaluate-base-gemini-2_5-pro:0.0.1"
}

# Option 1: use a system-defined metric
metric = "JSON_Match"           # one of: "JSON_Match", "LLMaaJ:Sem_Sim_1", "LLMaaJ:Sem_Sim_10", etc.
custom_metric_id = None
field_evaluation_metrics = None

# Option 2: use a custom LLM-as-a-judge metric (set metric=None and provide the ID)
# metric = None
# custom_metric_id = "<your-custom-metric-id>"
# field_evaluation_metrics = None

# Option 3: per-field evaluation metrics — requires response_format with json_schema in prompt template
# Supported metrics per field: "ExactMatch", "LLMaaJ:Sem_Sim_1"
# Set metric=None and custom_metric_id=None when using this option
# metric = None
# custom_metric_id = None
# field_evaluation_metrics = {
#     "urgency": "ExactMatch",
#     "sentiment": "LLMaaJ:Sem_Sim_1",
#     "categories.emergency_repair_services": "ExactMatch",
# }

# Optional: override model parameters per target model
# Supported params: temperature (0-1), max_tokens (int)
# Format: JSON string mapping model ID to parameter dict
model_params = None
# model_params = json.dumps({
#     "gemini-2.5-pro:001": {"temperature": 0.5, "max_tokens": 1024}
# })

# Optional: map prompt placeholder names to dataset field names
# Use when variable names in your prompt template differ from the dataset field names
# Format: JSON string mapping prompt variable -> dataset field
variable_mapping = None
# variable_mapping = json.dumps({"user_query": "input_text", "context": "background_info"})

# Optional: prototype mode — use as few as 3 samples for quick PoCs
# Not recommended for production — use 25+ samples for reliable results.
prototype_mode = "false"   # set to "true" to enable


## Push Local Prompt to Registry

In [14]:
base_template = load_prompt_template(base_prompt_template)
prompt_template_name_registry, _, prompt_template_version = base_prompt_template_registry.partition(":")
prompt = push_prompt_template(prompt_template=base_template,
                              prompt_template_name_registry=prompt_template_name_registry,
                              prompt_template_version=prompt_template_version,
                              scenario=scenario,
                              update=False
)

print(f"Prompt present in registry under id {prompt['id']}")

print('\n\n=== Base Prompt ===')
print_prompt_template(prompt["id"])

Prompt template evaluate-base already exists. Use update=True to update.

Prompt present in registry under id 6380ee92-1a61-4f5d-9467-4329675eac30

=== Base Prompt ===

╭──────────────────────────────────────────────── System Message ─────────────────────────────────────────────────╮
│ You are a helpful assistant                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── User Message ──────────────────────────────────────────────────╮
│ Giving the following message:                                                                                   │
│ ---                                                                                                             │
│ {{?input}}                                                                                                      │
│ ---                                                                                                             │
│ Extract and return a json with the follwoing keys and values:                                                   │
│ - "urgency" as one of `high`, `medium`, `low`                                                                   │
│ - "sentiment" as one of `negative`, `neutral`, `positive`                                                       │
│ - "categories" Create a dictionary with categories as keys and boolean values (True/False), where the value     │
│ indicates whether the category is one of the best matching support category tags from:                          │
│ `emergency_repair_services`, `routine_maintenance_requests`, `quality_and_safety_concerns`,                     │
│ `specialized_cleaning_services`, `general_inquiries`, `sustainability_and_environmental_practices`,             │
│ `training_and_support_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`,            │
│ `facility_management_issues`                                                                                    │
│ Your complete message should be a valid json string that can be read directly and only contain the keys         │
│ mentioned in the list above. Never enclose it in ```json...```, no newlines, no unnessacary whitespaces.        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Push Local Dataset to Object Store and Create Artifact

In [15]:
artifact, dataset_path = upload_dataset(
    secret=dataset_secret,
    local_path=dataset_local_path,
    remote_path=dataset_remote_path,
    expected_keys=base_template.placeholders,
    scenario=scenario,
    overwrite=True,
    allow_bucket_root=True
)

print(f"Dataset uploaded to {artifact.url}/{dataset_path} -> Artifact ID: {artifact.id}")


Dataset uploaded to ai://default/datasets/facility-train.json -> Artifact ID: 866bd88d-f983-44af-8e28-cce43bc62a5f

## Create Prompt Optimizer Config

In [16]:
configuration_id = create_config(
    metric=metric,
    custom_metric_id=custom_metric_id,
    reference_model=reference_model,
    targets=targets,
    dataset_path=dataset_path,
    scenario=scenario,
    prompt=prompt,
    artifact_id=artifact.id,
    model_params=model_params,
    variable_mapping=variable_mapping,
    prototype_mode=prototype_mode,
    field_evaluation_metrics=field_evaluation_metrics,
)


## Start Prompt Optimizer

In [17]:
response = client.ai_core_client.execution.create(
    configuration_id = configuration_id,
    resource_group = resource_group
)

execution_id = response.id
print('Execution started with ID:', execution_id)

Execution started with ID: edfce82fa76d3971

## Wait for Completion

In [18]:
import time
import sys

def wait_for_completion(execution_id, timeout=3600, initial_interval=120, pending_interval=40):
    """Poll until execution reaches COMPLETED or a terminal status."""
    start = time.time()
    last_status = None
    interval = initial_interval
    while time.time() - start < timeout:
        response = client.ai_core_client.execution.get(
            execution_id=execution_id, resource_group=resource_group, select="status"
        )
        status = response.status.name
        elapsed = int(time.time() - start)
        if status != last_status:
            print(f"[{elapsed}s] Status: {status}", flush=True)
            last_status = status
        if status == "COMPLETED":
            print("Optimization completed successfully.", flush=True)
            return
        elif status in {"DEAD", "STOPPED", "STOPPING"}:
            print(f"Execution ended with status: {status}. Use the debug cells below to investigate.", flush=True)
            return
        elif status == "UNKNOWN":
            interval = initial_interval
        else:
            interval = pending_interval
        time.sleep(interval)
    print(f"Timed out after {timeout}s. Last status: {status}", flush=True)

wait_for_completion(execution_id)

[0s] Status: UNKNOWN

[121s] Status: RUNNING

[896s] Status: COMPLETED

Optimization completed successfully.

## View Results

In [19]:
fetch_and_print_results(execution_id)

                         Score Summary                         
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━┓
┃ Model              ┃ Baseline ┃  Pre  ┃ Post  ┃ Improvement ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━┩
│ gpt-4o:2024-08-06  │  0.837   │   –   │   –   │      –      │
│ gemini-2.5-pro:001 │    –     │ 0.917 │ 0.960 │   ▲ +4.7%   │
└────────────────────┴──────────┴───────┴───────┴─────────────┘

  Evaluation Details —   
    gpt-4o:2024-08-06    
┏━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric     ┃ Baseline ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━┩
│ f1         │    0.837 │
│ tp         │      251 │
│ llm_p      │      300 │
│ recall     │    0.837 │
│ ground_p   │      300 │
│ precision  │    0.837 │
│ is_correct │       20 │
└────────────┴──────────┘

   Token Consumption —   
    gpt-4o:2024-08-06    
┏━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Metric        ┃ Value ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━┩
│ input_tokens  │ 6,385 │
│ output_tokens │ 2,400 │
│ num_requests  │    25 │
└───────────────┴───────┘

     Evaluation Details —     
      gemini-2.5-pro:001      
┏━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ Metric     ┃   Pre ┃  Post ┃
┡━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ f1         │ 0.917 │ 0.960 │
│ tp         │   275 │   288 │
│ llm_p      │   300 │   300 │
│ recall     │ 0.917 │ 0.960 │
│ ground_p   │   300 │   300 │
│ precision  │ 0.917 │ 0.960 │
│ is_correct │    25 │    25 │
└────────────┴───────┴───────┘

                                      Token Consumption — gemini-2.5-pro:001                                       
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric                  ┃                                                                                 Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ provider_request_counts │ [{'provider': {'model_name': 'gpt-5', 'model_params': {'reasoning_effort': 'medium'}, │
│                         │        'model_version': '2025-08-07'}, 'num_requests': 43, 'purpose': 'optimization', │
│                         │         'input_tokens': 182983, 'output_tokens': 122897}, {'provider': {'model_name': │
│                         │     'gpt-4o', 'model_params': {}, 'model_version': '2024-08-06'}, 'num_requests': 25, │
│                         │       'purpose': 'origin', 'input_tokens': 6385, 'output_tokens': 2400}, {'provider': │
│                         │         {'model_name': 'gemini-2.5-pro', 'model_params': {}, 'model_version': '001'}, │
│                         │    'num_requests': 480, 'purpose': 'target', 'input_tokens': 647854, 'output_tokens': │
│                         │            583451}, {'provider': {'model_name': 'gemini-2.5-pro', 'model_params': {}, │
│                         │               'model_version': '001'}, 'num_requests': 20, 'purpose': 'optimization', │
│                         │                                      'input_tokens': 157079, 'output_tokens': 60309}] │
└─────────────────────────┴───────────────────────────────────────────────────────────────────────────────────────┘

╭──────────────────────────────── System Message - Optimized — gemini-2.5-pro:001 ────────────────────────────────╮
│ You are an expert facility support message classifier.                                                          │
│                                                                                                                 │
│ Your objective is to classify incoming facility support messages by returning a single minified JSON object     │
│ with `urgency`, `sentiment`, and `categories` keys. You must follow the detailed definitions and rules provided │
│ below.                                                                                                          │
│                                                                                                                 │
│ Category Definitions (evaluate each independently and set as applicable):                                       │
│ - emergency_repair_services: Immediate repair needs for broken or failing facility systems requiring urgent     │
│ response.                                                                                                       │
│ - routine_maintenance_requests: Planned or non-urgent upkeep or routine service requests.                       │
│ - quality_and_safety_concerns: Reports of hazards, incidents, unsafe conditions, or compliance risks. Do not    │
│ set this for routine training or policy education unless an actual issue or hazard is described.                │
│ - specialized_cleaning_services: Requests for non-routine or specialized cleaning (e.g., deep cleaning,         │
│ post-construction, biohazard, high-reach, hazardous cleanup).                                                   │
│ - general_inquiries: Requests for information about services, pricing, capabilities, or availability without a  │
│ specific service action requested. Co-tagging rule: Set general_inquiries when the message primarily seeks      │
│ information; co-tag with a specific service category if the inquiry concerns that service’s                     │
│ information/availability rather than initiating work. Do not set general_inquiries when the message is          │
│ instructing to perform or schedule a service.                                                                   │
│ - sustainability_and_environmental_practices: Topics related to recycling, waste reduction, eco-friendly        │
│ products, environmental performance, or sustainability programs.                                                │
│ - training_and_support_requests: Requests to schedule or obtain training/support on equipment, cleaning         │
│ protocols, or related materials/resources.                                                                      │
│ - cleaning_services_scheduling: Scheduling, rescheduling, or canceling cleaning services, or asking for         │
│ calendar availability related to cleaning.                                                                      │
│ - customer_feedback_and_complaints: Expressions of praise, satisfaction, dissatisfaction, or complaints about   │
│ service quality or experience.                                                                                  │
│ - facility_management_issues: Non-emergency facility operations topics (e.g., space/access coordination,        │
│ utilities planning, process/policy issues) that are not immediate repairs. This can be co-tagged with emergency │
│ categories if the issue involves broader operational coordination beyond the immediate repair. Exclusions: Do   │
│ not set facility_management_issues for routine maintenance or cleaning requests, including their scheduling;    │
│ use the corresponding maintenance/cleaning categories instead.                                                  │
│                                                                                                                 │
│ Sentiment Rules:                                      

╭───────────────────────────────── User Message - Optimized — gemini-2.5-pro:001 ─────────────────────────────────╮
│ # Instructions                                                                                                  │
│ - Analyze the user message provided in the `<message>` tags.                                                    │
│ - Classify the message by its urgency, sentiment, and applicable categories based on the detailed definitions   │
│ and rules provided in your system prompt.                                                                       │
│ - Your output must be a single, minified JSON object.                                                           │
│                                                                                                                 │
│ # Reasoning Steps                                                                                               │
│ 1.  Determine the `urgency` (high, medium, or low) by applying the urgency rules from the system prompt.        │
│ 2.  Determine the `sentiment` (positive, negative, or neutral) by applying the sentiment rules from the system  │
│ prompt.                                                                                                         │
│ 3.  For each category defined in the system prompt, independently evaluate if it applies to the message and set │
│ its value to `true` or `false`.                                                                                 │
│ 4.  Construct the final JSON object ensuring it adheres to the output format.                                   │
│                                                                                                                 │
│ # Output Format                                                                                                 │
│ - Return only a single-line, minified JSON object with no newlines or extraneous whitespace.                    │
│ - The JSON object must contain exactly three keys: "urgency", "sentiment", and "categories".                    │
│ - The "urgency" value must be one of: "high", "medium", "low".                                                  │
│ - The "sentiment" value must be one of: "positive", "negative", "neutral".                                      │
│ - The "categories" value must be an object containing a key for every category defined in the system prompt,    │
│ with each value being a JSON boolean (`true` or `false`).                                                       │
│                                                                                                                 │
│ # Context                                                                                                       │
│ <message>                                                                                                       │
│ {{?input}}                                                                                                      │
│ </message>                                                                                                      │
│                                                                                                                 │
│ Return only the JSON object as specified.                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯